## Учёт заказов в кофейне

В небольшой кофейне сотрудники принимают заказы от клиентов.
Нужно сделать консольную программу для учёта заказов, их статусов и оплаты.

### Что должна уметь программа

Меню:
```txt
1. Добавить заказ
2. Показать все заказы
3. Изменить статус заказа
4. Показать активные заказы
5. Найти заказы по имени клиента
6. Отметить заказ как оплаченный
7. Показать неоплаченные заказы
8. Показать общую выручку
0. Выход
```

### Данные одного заказа
```txt
Номер заказа
Имя клиента
Состав заказа
Стоимость
Статус
Оплачен / не оплачен
Приоритет
```

### Статусы заказа
```txt
новый
готовится
готов
выдан
отменён
```

### Приоритет заказа
```txt
обычный
срочный
```

### Пример заказа
```txt
№1
Клиент: Анна
Заказ: Капучино и круассан
Стоимость: 450 руб.
Статус: новый
Оплата: не оплачен
Приоритет: обычный
```

### Обязательные условия

1. Номер заказа создаётся автоматически.
2. При добавлении заказа статус всегда новый.
3. При добавлении заказ всегда считается не оплачен.
4. Стоимость заказа должна быть больше 0.
5. Можно изменить статус только на:
   - новый
   - готовится
   - готов
   - выдан
   - отменён
6. Заказ нельзя сделать выданным, если он не оплачен.
7. Отменённый заказ нельзя изменить на другой статус.
8. Активные заказы — это заказы со статусом новый, готовится или готов.
9. Общая выручка считается только по оплаченным заказам, которые не были отменены.
10. При поиске по имени клиента программа должна показывать все заказы этого клиента.
11. Срочные заказы при выводе активных заказов должны показываться первыми.


### Требования к ООП и SOLID

Решение должно быть реализовано в объектно-ориентированном стиле с соблюдением принципов SOLID:

1. Отдельный класс для заказа.
2. Отдельный сервис для управления заказами.
3. Отдельный класс для работы с консольным меню.
4. Отдельная логика проверки статусов и оплаты.
5. Классы не должны брать на себя лишние обязанности.
6. Зависимости между частями программы должны быть минимальными и понятными.

In [ ]:
from enum import Enum


class Status(Enum):
    NEW = "новый"
    COOKING = "готовится"
    READY = "готов"
    DONE = "выдан"
    CANCELED = "отменён"


class Priority(Enum):
    NORMAL = "обычный"
    URGENT = "срочный"


class Order:
    _id_counter = 1

    def __init__(self, customer_name, items, price, priority):
        if price <= 0:
            raise ValueError("Стоимость должна быть больше 0")

        self.id = Order._id_counter
        Order._id_counter += 1

        self.customer_name = customer_name
        self.items = items
        self.price = price
        self.status = Status.NEW
        self.paid = False
        self.priority = priority

    def __str__(self):
        return (
            f"№{self.id}\n"
            f"Клиент: {self.customer_name}\n"
            f"Заказ: {self.items}\n"
            f"Стоимость: {self.price} руб.\n"
            f"Статус: {self.status.value}\n"
            f"Оплата: {'оплачен' if self.paid else 'не оплачен'}\n"
            f"Приоритет: {self.priority.value}\n"
        )


class OrderValidator:

    @staticmethod
    def can_change_status(order, new_status):
        if order.status == Status.CANCELED:
            return False, "Отменённый заказ нельзя изменить"

        if new_status == Status.DONE and not order.paid:
            return False, "Нельзя выдать заказ без оплаты"

        return True, ""

    @staticmethod
    def is_active(order):
        return order.status in {Status.NEW, Status.COOKING, Status.READY}

    @staticmethod
    def is_revenue(order):
        return order.paid and order.status != Status.CANCELED


class OrderService:
    def __init__(self):
        self.orders = []

    def add_order(self, customer_name, items, price, priority):
        order = Order(customer_name, items, price, priority)
        self.orders.append(order)

    def get_all_orders(self):
        return self.orders

    def find_by_id(self, order_id):
        for order in self.orders:
            if order.id == order_id:
                return order
        return None

    def change_status(self, order_id, new_status):
        order = self.find_by_id(order_id)
        if not order:
            return "Заказ не найден"

        allowed, message = OrderValidator.can_change_status(order, new_status)
        if not allowed:
            return message

        order.status = new_status
        return "Статус обновлён"

    def mark_paid(self, order_id):
        order = self.find_by_id(order_id)
        if not order:
            return "Заказ не найден"

        order.paid = True
        return "Заказ оплачен"

    def get_active_orders(self):
        active = [o for o in self.orders if OrderValidator.is_active(o)]
        return sorted(active, key=lambda x: x.priority == Priority.NORMAL)

    def find_by_customer(self, name):
        return [o for o in self.orders if o.customer_name.lower() == name.lower()]

    def get_unpaid_orders(self):
        return [o for o in self.orders if not o.paid]

    def get_revenue(self):
        return sum(o.price for o in self.orders if OrderValidator.is_revenue(o))


class ConsoleMenu:
    def __init__(self):
        self.service = OrderService()

    def run(self):
        while True:
            print("""1. Добавить заказ
                     2. Показать все заказы
                     3. Изменить статус заказа
                     4. Показать активные заказы
                     5. Найти заказы по имени клиента
                     6. Отметить заказ как оплаченный
                     7. Показать неоплаченные заказы
                     8. Показать общую выручку
                     0. Выход""")

            choice = input("Выберите пункт: ")

            if choice == "1":
                self.add_order()
            elif choice == "2":
                self.show_orders(self.service.get_all_orders())
            elif choice == "3":
                self.change_status()
            elif choice == "4":
                self.show_orders(self.service.get_active_orders())
            elif choice == "5":
                self.find_orders()
            elif choice == "6":
                self.mark_paid()
            elif choice == "7":
                self.show_orders(self.service.get_unpaid_orders())
            elif choice == "8":
                print("Выручка:", self.service.get_revenue(), "руб.")
            elif choice == "0":
                break
            else:
                print("Неверный выбор")

    def add_order(self):
        name = input("Имя клиента: ")
        items = input("Состав заказа: ")
        price = float(input("Стоимость: "))
        priority_input = input("Приоритет (1-обычный, 2-срочный): ")

        priority = Priority.URGENT if priority_input == "2" else Priority.NORMAL

        try:
            self.service.add_order(name, items, price, priority)
            print("Заказ добавлен")
        except ValueError as e:
            print(e)

    def change_status(self):
        order_id = int(input("ID заказа: "))
        print("Статусы:")
        for s in Status:
            print(s.value)

        status_input = input("Введите статус: ")

        for s in Status:
            if s.value == status_input:
                print(self.service.change_status(order_id, s))
                return

        print("Неверный статус")

    def mark_paid(self):
        order_id = int(input("ID заказа: "))
        print(self.service.mark_paid(order_id))

    def find_orders(self):
        name = input("Имя клиента: ")
        orders = self.service.find_by_customer(name)
        self.show_orders(orders)

    def show_orders(self, orders):
        if not orders:
            print("Нет заказов")
            return

        for order in orders:
            print(order)
            print("-" * 20)


if __name__ == "__main__":
    menu = ConsoleMenu()
    menu.run()


1. Добавить заказ
2. Показать все заказы
3. Изменить статус заказа
4. Показать активные заказы
5. Найти заказы по имени клиента
6. Отметить заказ как оплаченный
7. Показать неоплаченные заказы
8. Показать общую выручку
0. Выход

